In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName('new_app').config('spark.executor.memory', '4g').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/23 11:17:18 WARN Utils: Your hostname, LAPTOP-C6UDPGMF, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/23 11:17:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 11:17:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.csv('fmcg_marketing.csv', header = True)
df.show(5)

26/04/23 11:17:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------+----------+----+-------+-----+----------+-------------+---------+-----------+-------------+-------------+-------------+-----------------+-----------------+-----------+------------------+----------+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+
|        Order_ID|Order_Date|Year|Quarter|Month|Month_Name|       Region|  Country|       City| Sales_Person|Customer_Type|Sales_Channel|   Promotion_Type| Product_Category|      Brand|      Product_Name|       SKU|Units_Sold|Unit_Price_USD|Discount_Pct|Gross_Sales_USD|Marketing_Spend_USD|COGS_USD|Logistics_Cost_USD|Net_Revenue_USD|Profit_USD|Profit_Margin_Pct|
+----------------+----------+----+-------+-----+----------+-------------+---------+-----------+-------------+-------------+-------------+-----------------+-----------------+-----------+------------------+----------+----------+--------------+------------+---------------+--

In [4]:
df.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Quarter: string (nullable = true)
 |-- Month: string (nullable = true)
 |-- Month_Name: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Sales_Person: string (nullable = true)
 |-- Customer_Type: string (nullable = true)
 |-- Sales_Channel: string (nullable = true)
 |-- Promotion_Type: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- SKU: string (nullable = true)
 |-- Units_Sold: string (nullable = true)
 |-- Unit_Price_USD: string (nullable = true)
 |-- Discount_Pct: string (nullable = true)
 |-- Gross_Sales_USD: string (nullable = true)
 |-- Marketing_Spend_USD: string (nullable = true)
 |-- COGS_USD: string (nullable = true)
 |-- Logistics_Cost_USD: string 

In [5]:
df.show(4)
# No. of null values in each column
df.select([count(when(col(c).isNull(), c)).alias(c+'_null') for c in df.columns]).show()

+----------------+----------+----+-------+-----+----------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+------------------+----------+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+
|        Order_ID|Order_Date|Year|Quarter|Month|Month_Name|       Region|  Country|       City|Sales_Person|Customer_Type|Sales_Channel|   Promotion_Type|Product_Category|      Brand|      Product_Name|       SKU|Units_Sold|Unit_Price_USD|Discount_Pct|Gross_Sales_USD|Marketing_Spend_USD|COGS_USD|Logistics_Cost_USD|Net_Revenue_USD|Profit_USD|Profit_Margin_Pct|
+----------------+----------+----+-------+-----+----------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+------------------+----------+----------+--------------+------------+---------------+--------

+-------------+---------------+---------+------------+----------+---------------+-----------+------------+---------+-----------------+------------------+------------------+-------------------+---------------------+----------+-----------------+--------+---------------+-------------------+-----------------+--------------------+------------------------+-------------+-----------------------+--------------------+---------------+----------------------+
|Order_ID_null|Order_Date_null|Year_null|Quarter_null|Month_null|Month_Name_null|Region_null|Country_null|City_null|Sales_Person_null|Customer_Type_null|Sales_Channel_null|Promotion_Type_null|Product_Category_null|Brand_null|Product_Name_null|SKU_null|Units_Sold_null|Unit_Price_USD_null|Discount_Pct_null|Gross_Sales_USD_null|Marketing_Spend_USD_null|COGS_USD_null|Logistics_Cost_USD_null|Net_Revenue_USD_null|Profit_USD_null|Profit_Margin_Pct_null|
+-------------+---------------+---------+------------+----------+---------------+-----------+-----

In [6]:
# No. of unique values in each column
df.select([count_distinct(col(c)).alias(c) for c in df.columns]).show()

+--------+----------+----+-------+-----+----------+------+-------+----+------------+-------------+-------------+--------------+----------------+-----+------------+---+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+
|Order_ID|Order_Date|Year|Quarter|Month|Month_Name|Region|Country|City|Sales_Person|Customer_Type|Sales_Channel|Promotion_Type|Product_Category|Brand|Product_Name|SKU|Units_Sold|Unit_Price_USD|Discount_Pct|Gross_Sales_USD|Marketing_Spend_USD|COGS_USD|Logistics_Cost_USD|Net_Revenue_USD|Profit_USD|Profit_Margin_Pct|
+--------+----------+----+-------+-----+----------+------+-------+----+------------+-------------+-------------+--------------+----------------+-----+------------+---+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+
|   18240|      1096|   3|      4|   12|        12| 

In [7]:
df.show(4)

+----------------+----------+----+-------+-----+----------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+------------------+----------+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+
|        Order_ID|Order_Date|Year|Quarter|Month|Month_Name|       Region|  Country|       City|Sales_Person|Customer_Type|Sales_Channel|   Promotion_Type|Product_Category|      Brand|      Product_Name|       SKU|Units_Sold|Unit_Price_USD|Discount_Pct|Gross_Sales_USD|Marketing_Spend_USD|COGS_USD|Logistics_Cost_USD|Net_Revenue_USD|Profit_USD|Profit_Margin_Pct|
+----------------+----------+----+-------+-----+----------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+------------------+----------+----------+--------------+------------+---------------+--------

In [8]:
id_cols = ['Order_ID', 'SKU']
date_cols = ['Order_Date', 'Year', 'Quarter', 'Month', 'Month_Name']
categorical_cols = ['Region', 'Country', 'City', 'Sales_Person', 'Customer_Type', 'Sales_Channel', 'Promotion_Type', 'Product_Category','Brand', 'Product_Name']
float_cols = ['Unit_Price_USD', 'Discount_Pct', 'Gross_Sales_USD', 'Marketing_Spend_USD', 'COGS_USD', 'Logistics_Cost_USD', 'Net_Revenue_USD', 'Profit_USD', 'Profit_Margin_Pct']
int_cols = ['Units_Sold']

In [9]:
print(len(df.columns))
print(len(id_cols))
print(len(date_cols))
print(len(categorical_cols))
print(len(float_cols))
print(len(int_cols))

27
2
5
10
9
1


<h3>Clean String issues globally</h3>

In [10]:
df = df.select([trim(col(c)).alias(c) for c in df.columns])
df = df.select([lower(col(c)).alias(c) for c in df.columns])

<h3>Handling Date related features </h3>

In [11]:
df = df.withColumn('Order_Date', to_timestamp(col('Order_Date'), 'yyyy-MM-dd'))

In [12]:
# Validating derived columns
sample = df.select(col('Order_Date'), col('Year'), year(col('Order_Date')).alias('Derived_Year'), when(col('Year') != col('Derived_Year'), 1).otherwise(0).alias('Year - Derived_Year'))
sample.filter(col('Year - Derived_Year') == 1).show()

+----------+----+------------+-------------------+
|Order_Date|Year|Derived_Year|Year - Derived_Year|
+----------+----+------------+-------------------+
+----------+----+------------+-------------------+



In [13]:
df = df.drop('Year', 'Quarter', 'Month', 'Month_Name')
# df.select(date_format(col('Order_Date'), "MMMM").alias('month_name')).show()

df = df.withColumn('Year', year(col('Order_Date'))) \
        .withColumn('Month', month(col('Order_Date'))) \
        .withColumn('Quarter', quarter(col('Order_Date'))) \
        .withColumn('Month_Name', date_format(col('Order_Date'), 'MMMM')) \
        .withColumn('Date', dayofmonth('Order_Date'))
df.show(3)

+----------------+-------------------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+----------------+----------+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+----+-----+-------+----------+----+
|        Order_ID|         Order_Date|       Region|  Country|       City|Sales_Person|Customer_Type|Sales_Channel|   Promotion_Type|Product_Category|      Brand|    Product_Name|       SKU|Units_Sold|Unit_Price_USD|Discount_Pct|Gross_Sales_USD|Marketing_Spend_USD|COGS_USD|Logistics_Cost_USD|Net_Revenue_USD|Profit_USD|Profit_Margin_Pct|Year|Month|Quarter|Month_Name|Date|
+----------------+-------------------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+----------------+----------+----------+--------------+------------+---------------+-------

<h3>Numeric features handling </h3>

In [14]:
df.show(3)

+----------------+-------------------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+----------------+----------+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+----+-----+-------+----------+----+
|        Order_ID|         Order_Date|       Region|  Country|       City|Sales_Person|Customer_Type|Sales_Channel|   Promotion_Type|Product_Category|      Brand|    Product_Name|       SKU|Units_Sold|Unit_Price_USD|Discount_Pct|Gross_Sales_USD|Marketing_Spend_USD|COGS_USD|Logistics_Cost_USD|Net_Revenue_USD|Profit_USD|Profit_Margin_Pct|Year|Month|Quarter|Month_Name|Date|
+----------------+-------------------+-------------+---------+-----------+------------+-------------+-------------+-----------------+----------------+-----------+----------------+----------+----------+--------------+------------+---------------+-------

In [15]:
df = df.select(*[col(c).cast('float').alias(c) if c in float_cols else col(c) for c in df.columns])
df = df.select(*[col(c).cast('int').alias(c) if c in int_cols else col(c) for c in df.columns])

In [16]:
df.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: timestamp (nullable = true)
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Sales_Person: string (nullable = true)
 |-- Customer_Type: string (nullable = true)
 |-- Sales_Channel: string (nullable = true)
 |-- Promotion_Type: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- SKU: string (nullable = true)
 |-- Units_Sold: integer (nullable = true)
 |-- Unit_Price_USD: float (nullable = true)
 |-- Discount_Pct: float (nullable = true)
 |-- Gross_Sales_USD: float (nullable = true)
 |-- Marketing_Spend_USD: float (nullable = true)
 |-- COGS_USD: float (nullable = true)
 |-- Logistics_Cost_USD: float (nullable = true)
 |-- Net_Revenue_USD: float (nullable = true)
 |-- Profit_USD: float (nullable = true)
 |-- Profit_Margin_Pct: float (nullable = true)
 |-

In [17]:
df.select([count(when(col(c).isNull(), c)).alias(c+'_null') for c in df.columns]).show()

# checking whether nulls increased after type conversion

+-------------+---------------+-----------+------------+---------+-----------------+------------------+------------------+-------------------+---------------------+----------+-----------------+--------+---------------+-------------------+-----------------+--------------------+------------------------+-------------+-----------------------+--------------------+---------------+----------------------+---------+----------+------------+---------------+---------+
|Order_ID_null|Order_Date_null|Region_null|Country_null|City_null|Sales_Person_null|Customer_Type_null|Sales_Channel_null|Promotion_Type_null|Product_Category_null|Brand_null|Product_Name_null|SKU_null|Units_Sold_null|Unit_Price_USD_null|Discount_Pct_null|Gross_Sales_USD_null|Marketing_Spend_USD_null|COGS_USD_null|Logistics_Cost_USD_null|Net_Revenue_USD_null|Profit_USD_null|Profit_Margin_Pct_null|Year_null|Month_null|Quarter_null|Month_Name_null|Date_null|
+-------------+---------------+-----------+------------+---------+------------

In [18]:
for i in categorical_cols:
    df.select(i).distinct().show()


+-------------+
|       Region|
+-------------+
|north america|
|       europe|
|south america|
|      oceania|
|         asia|
+-------------+

+-----------+
|    Country|
+-----------+
|  australia|
|        uae|
|new zealand|
|   thailand|
|   colombia|
|      india|
|         uk|
|  indonesia|
|     brazil|
|     canada|
|     france|
|    germany|
|      italy|
|      chile|
|      spain|
|        usa|
|     mexico|
+-----------+

+----------+
|      City|
+----------+
|     delhi|
|  new york|
|   atlanta|
|  brisbane|
| vancouver|
|   chicago|
|     paris|
|  santiago|
|   toronto|
| melbourne|
|    dallas|
|valparaiso|
| monterrey|
|  valencia|
|    berlin|
|   bangkok|
|      lyon|
|     turin|
|    london|
|   hamburg|
+----------+
only showing top 20 rows
+---------------+
|   Sales_Person|
+---------------+
| javier morales|
|charlotte ellis|
|     rohan iyer|
| farah siddiqui|
|   elena dupont|
|   sharon patel|
|     mia sutton|
| bruno carvalho|
|  camila torres|
|    ma

In [19]:
unwanted_cols = ['SKU']
df = df.drop(*unwanted_cols)

In [20]:
df = df.dropDuplicates()
df.count()

18240

In [21]:
# df.select('Sales_Person').distinct().show()
df.select('Product_Category').distinct().show()

# df.filter(col('Sales_Person') == 'mia sutton').select('Sales_Channel').distinct().show()


+-----------------+
| Product_Category|
+-----------------+
|        beverages|
|dairy & breakfast|
|    personal care|
|        household|
|           snacks|
+-----------------+



<h3>Silver to Gold </h3>

In [23]:
df.write.mode('overwrite').saveAsTable('sil_table')

<h4>dim_location</h4>

In [ ]:
df.select('Region', 'Country', 'City').count()

18240

In [ ]:
df.select('Region', 'Country', 'City').distinct().count()

48

In [24]:
dim_location = df.select('Region', 'Country', 'City').distinct()

w = Window.orderBy(lit(1))
dim_location = dim_location.withColumn('Location_Id', row_number().over(w))

dim_location.count()

48

In [25]:

dim_location_df = spark.sql('SELECT Region, Country, City, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS Location_Id FROM ' \
'(SELECT DISTINCT Region, Country, City FROM sil_table)')
dim_location_df.count()

48

<h3>Hashing Ids (Sha256) </h3>

In [ ]:
dim_location.withColumn("Location_Id", sha2(concat_ws("||", 'Region', 'Country', 'City'), 256)).show()

+-------------+---------+-----------+--------------------+
|       Region|  Country|       City|         Location_Id|
+-------------+---------+-----------+--------------------+
|north america|   mexico|guadalajara|f672b93005909e861...|
|north america|   mexico|  monterrey|4dadacef1e99d3e25...|
|north america|      usa|   new york|0fdd0ce91dec4ab5f...|
|north america|   mexico|mexico city|a1bfe39475d11d111...|
|      oceania|australia|  melbourne|30c9980bbceba70a2...|
|       europe|   france|  marseille|0ea2385fe5b3971cb...|
|         asia|    india|  hyderabad|82385143bc81da06b...|
|      oceania|australia|     sydney|61400aa53941c503f...|
|south america| colombia|     bogota|797d84d02dd85f6a8...|
|north america|   canada|   montreal|3f59958fbaa1f58fd...|
|       europe|    italy|      milan|56bec510dabc6a0b7...|
|         asia|indonesia|   surabaya|66441158cbdb19ea4...|
|       europe|  germany|     munich|ea5e1288e0cf29ad3...|
|       europe|  germany|    hamburg|eeea4fbdf668cc99e..

<h4>dim_salesperson</h4>

In [ ]:
dim_salesperson = df.select('Sales_Person').distinct()
w = Window.orderBy(lit(1))
dim_salesperson = dim_salesperson.withColumn('Salesperson_Id', row_number().over(w))
print(dim_salesperson.count())

# sql
dim_salesperson_df = spark.sql('SELECT Sales_Person, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS Salesperson_Id ' \
        'FROM (SELECT DISTINCT Sales_Person FROM sil_table) unique_list'         
)
print(dim_salesperson_df.count())

# cte sql
dim_salesperson_df = spark.sql('WITH unique_salesperson AS( ' \
    '   SELECT Sales_Person FROM sil_table GROUP BY Sales_Person ' \
    # GROUP BY is often slightly faster than DISTINCT for deduplication in Spark
    ') ' \
    'SELECT Sales_Person, ROW_NUMBER() OVER (ORDER BY(SELECT NULL)) AS Salesperson_Id ' \
    'FROM unique_salesperson'
)
print(dim_salesperson_df.count())

36
36
36


<h4>dim_customer</h4>

In [ ]:
w = Window.orderBy(lit(1))
dim_customer = df.select('Customer_Type').distinct()
dim_customer = dim_customer.withColumn('Customertype_Id', row_number().over(w))
dim_customer.show()

# sql
dim_customer_df = spark.sql('WITH unique_customertype AS (' \
'SELECT Customer_Type FROM sil_table GROUP BY Customer_Type' \
')' \
'SELECT Customer_Type, ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS Customertype_Id FROM unique_customertype')
dim_customer_df.show()

26/04/23 06:52:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 0

+-------------+---------------+
|Customer_Type|Customertype_Id|
+-------------+---------------+
|          b2b|              1|
|          b2c|              2|
+-------------+---------------+



26/04/23 06:52:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+---------------+
|Customer_Type|Customertype_Id|
+-------------+---------------+
|          b2b|              1|
|          b2c|              2|
+-------------+---------------+



26/04/23 06:52:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


<h3>dim_channel</h3>

In [ ]:
dim_channel = df.select('Sales_Channel').distinct()
dim_channel.show()

w = Window.orderBy(lit(1))
dim_channel = dim_channel.withColumn('Channel_Id', row_number().over(w))
dim_channel.show()

# sql
dim_channel_df = spark.sql('WITH unique_channels AS(' \
'SELECT Sales_Channel FROM sil_table GROUP BY Sales_Channel' \
')' \
'SELECT Sales_Channel, ROW_NUMBER() OVER(ORDER BY(SELECT NULL)) AS Channel_Id FROM unique_channels')

dim_channel_df.show()

+-------------+
|Sales_Channel|
+-------------+
|       online|
|  distributor|
|    wholesale|
| modern trade|
+-------------+



26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 0

+-------------+----------+
|Sales_Channel|Channel_Id|
+-------------+----------+
|       online|         1|
|  distributor|         2|
|    wholesale|         3|
| modern trade|         4|
+-------------+----------+



26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------+----------+
|Sales_Channel|Channel_Id|
+-------------+----------+
|       online|         1|
|  distributor|         2|
|    wholesale|         3|
| modern trade|         4|
+-------------+----------+



In [ ]:
dim_promotion = df.select('Promotion_Type').distinct()
w = Window.orderBy(lit(1))
dim_promotion = dim_promotion.withColumn('Promotion_Id', row_number().over(w))
dim_promotion.show()

# sql
dim_promotion_df = spark.sql('WITH unique_prom AS (SELECT Promotion_Type FROM sil_table GROUP BY Promotion_Type)' \
'SELECT Promotion_Type, ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) AS Promotion_Id FROM unique_prom')

dim_promotion_df.show()

26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 0

+------------------+------------+
|    Promotion_Type|Promotion_Id|
+------------------+------------+
|introductory offer|           1|
|    flash discount|           2|
| festival campaign|           3|
|      bundle offer|           4|
|  loyalty cashback|           5|
| seasonal campaign|           6|
|          no promo|           7|
+------------------+------------+

+------------------+------------+
|    Promotion_Type|Promotion_Id|
+------------------+------------+
|introductory offer|           1|
|    flash discount|           2|
| festival campaign|           3|
|      bundle offer|           4|
|  loyalty cashback|           5|
| seasonal campaign|           6|
|          no promo|           7|
+------------------+------------+



26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [ ]:
dim_product = df.select('Product_Name', 'Product_Category', 'Brand').distinct()
w = Window.orderBy(lit(1))
dim_product = dim_product.withColumn('Product_Id', row_number().over(w))
print(dim_product.count())

# sql
dim_product_df = spark.sql('WITH unique_pro AS (' \
'SELECT Product_Name, Product_Category, Brand FROM sil_table GROUP BY Product_Name, Product_Category, Brand' \
') ' \
'SELECT ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) AS Product_Id, Product_Name, Product_Category, Brand ' \
'FROM unique_pro'
)

# dim_product_df = spark.sql("""
#     WITH unique_pro AS (
#         SELECT Product_Name, Product_Category, Brand 
#         FROM sil_table 
#         GROUP BY Product_Name, Product_Category, Brand
#     ) 
#     SELECT 
#         ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) AS Product_Id, 
#         Product_Name, 
#         Product_Category, 
#         Brand 
#     FROM unique_pro
# """)

print(dim_product_df.count())


30
30


In [ ]:
dim_date = df.select('Order_Date', 'Year', 'Month', 'Quarter', 'Month_Name').distinct()
w = Window.orderBy(lit(1))
dim_date = dim_date.withColumn('Date_Id', row_number().over(w))
print(dim_date.count())

# sql
dim_date_df = spark.sql('WITH unique_date AS (' \
'SELECT DISTINCT Order_Date, Year, Month, Quarter, Month_Name FROM sil_table' \
')' \
'SELECT ROW_NUMBER() OVER(ORDER BY (SELECT NULL)) AS Date_Id, Order_Date, Year, Quarter, Month, Month_Name FROM unique_date')
print(dim_date_df.count())

1096
1096


In [ ]:
gold_fact = df.join(
    dim_location_df, 
    on = ['Region', 'Country', 'City'],
    how = 'left'
)
gold_fact = gold_fact.drop('Region', 'Country', 'City')

In [ ]:
gold_fact = gold_fact.join(
    dim_salesperson_df,
    on = ['Sales_Person'],
    how = 'left'
)
gold_fact = gold_fact.drop('Sales_Person')

In [ ]:
gold_fact = gold_fact.join(
    dim_customer_df,
    on = 'Customer_Type',
    how = 'left'
)
gold_fact = gold_fact.drop('Customer_Type')

In [ ]:
gold_fact = gold_fact.join(
    dim_channel_df,
    on = 'Sales_Channel',
    how = 'left'
)
gold_fact = gold_fact.drop('Sales_Channel')

In [ ]:
gold_fact = gold_fact.join(
    dim_promotion_df,
    on = 'Promotion_Type',
    how = 'left'
)
gold_fact = gold_fact.drop('Promotion_Type')

In [ ]:
gold_fact = gold_fact.join(
    dim_product_df,
    on = ['Product_Name', 'Product_Category', 'Brand'],
    how = 'left'
)
gold_fact = gold_fact.drop('Product_Name', 'Product_Category', 'Brand')


In [ ]:
gold_fact = gold_fact.join(
    dim_date,
    on = ['Order_Date', 'Year', 'Month', 'Quarter', 'Month_Name'],
    how = 'left'
)
gold_fact = gold_fact.drop('Order_Date', 'Year', 'Month', 'Quarter', 'Month_Name')

In [ ]:
gold_fact.show()

26/04/23 06:52:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 06:52:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/23 0

+----------------+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+----+-----------+--------------+---------------+----------+------------+----------+-------+
|        Order_ID|Units_Sold|Unit_Price_USD|Discount_Pct|Gross_Sales_USD|Marketing_Spend_USD|COGS_USD|Logistics_Cost_USD|Net_Revenue_USD|Profit_USD|Profit_Margin_Pct|Date|Location_Id|Salesperson_Id|Customertype_Id|Channel_Id|Promotion_Id|Product_Id|Date_Id|
+----------------+----------+--------------+------------+---------------+-------------------+--------+------------------+---------------+----------+-----------------+----+-----------+--------------+---------------+----------+------------+----------+-------+
|fmcg-2023-000277|       367|          5.89|        8.84|        2161.63|             189.48| 1133.57|            147.05|        1970.54|    500.44|             25.4|  30|         20|            35|              1|         4| 

AssertionError: all exprs should be Column